# 📚 RAG 시스템 (Retrieval-Augmented Generation)

> **학습 목표**: ChromaDB + LangChain으로 제조 문서 기반 지능형 QA 시스템을 구축합니다.

---

## 📋 학습 내용

1. ✅ 문서 로딩 및 전처리 (RecursiveCharacterTextSplitter)
2. ✅ 한국어 임베딩 모델 (KoSRoBERTa)
3. ✅ 벡터 DB 구축 (ChromaDB Persistent Storage)
4. ✅ Semantic Search (Top-K Retrieval)
5. ✅ RAG 파이프라인 (LangChain Integration)
6. ✅ 하이브리드 검색 (BM25 + Semantic)
7. ✅ RAG 평가 (답변 품질, Context Relevance)

**소요 시간**: 약 50분  
**난이도**: ⭐⭐⭐⭐ (고급)  
**사전 지식**: NLP 기초, 벡터 검색, LLM 개념

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# LangChain Core
try:
    from langchain.document_loaders import DirectoryLoader, TextLoader
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain.embeddings import HuggingFaceEmbeddings
    from langchain.vectorstores import Chroma
    from langchain.chains import RetrievalQA
    from langchain.prompts import PromptTemplate
    print("✅ LangChain 로드 완료!")
except ImportError:
    print("⚠️ LangChain 설치 필요: pip install langchain chromadb")

# Sentence Transformers
try:
    from sentence_transformers import SentenceTransformer
    print("✅ Sentence Transformers 로드 완료!")
except ImportError:
    print("⚠️ Sentence Transformers 설치 필요: pip install sentence-transformers")

# BM25 (하이브리드 검색)
try:
    from rank_bm25 import BM25Okapi
    print("✅ BM25 로드 완료!")
except ImportError:
    print("⚠️ BM25 설치 필요: pip install rank-bm25")

# 데이터 분석
import pandas as pd
import numpy as np

# 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 유틸리티
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

print("\n✅ 모든 라이브러리 로드 완료!")

## 📂 Step 2: 제조 문서 준비

**샘플 문서**: 예지보전, 품질관리, 불량 분류 등 제조 도메인 문서

In [ ]:
# ============================================================
# KAMP 실제 데이터 로드 (가이드북 PDF → RAG 문서)
# Fallback: 런타임 생성 마크다운 문서
# ============================================================

from pathlib import Path

# KAMP 가이드북 PDF 경로
kamp_data_dir = Path('../../dataset/part3-1')
kamp_pdf_files = [
    kamp_data_dir / '가이드북_시지트로닉스.pdf',
    kamp_data_dir / '가이드북_인스틸(주).pdf',
    kamp_data_dir / '가이드북_(주)성보산업.pdf',
]

# KAMP 게시물 JSON 경로 (추가 문서 소스)
kamp_json_dir = kamp_data_dir / '게시물'

documents = []
data_source = None

# ---- 1차 시도: KAMP PDF 로드 ----
pdf_loaded = False
try:
    from langchain.document_loaders import PyPDFLoader
    
    existing_pdfs = [p for p in kamp_pdf_files if p.exists()]
    if existing_pdfs:
        print(f"📂 KAMP 가이드북 PDF 발견: {len(existing_pdfs)}개")
        print(f"   경로: {kamp_data_dir.resolve()}\n")
        
        for pdf_path in existing_pdfs:
            try:
                loader = PyPDFLoader(str(pdf_path))
                pdf_docs = loader.load()
                documents.extend(pdf_docs)
                print(f"   ✅ {pdf_path.name}: {len(pdf_docs)} 페이지 로드")
            except Exception as e:
                print(f"   ⚠️ {pdf_path.name} 로드 실패: {e}")
        
        if documents:
            pdf_loaded = True
            data_source = 'KAMP PDF'
            print(f"\n✅ KAMP PDF에서 총 {len(documents)}개 문서 로드 완료")
except ImportError:
    print("⚠️ PyPDFLoader 사용 불가 (pip install pypdf 필요)")

# ---- 2차 시도: KAMP JSON을 문서로 변환 ----
if not pdf_loaded and kamp_json_dir.exists():
    import json as _json
    from langchain.docstore.document import Document
    
    json_files = sorted(kamp_json_dir.glob('*.json'))
    if json_files:
        print(f"\n📂 KAMP 게시물 JSON에서 문서 생성: {len(json_files)}개 파일")
        
        for jf in json_files:
            try:
                with open(jf, 'r', encoding='utf-8') as f:
                    data = _json.load(f)
                
                # JSON 내용을 구조화된 텍스트 문서로 변환
                company = data.get('업체명', '알 수 없음')
                title = data.get('제목', '')
                industry = data.get('수집업종', '')
                purpose = data.get('수집목적', '')
                equipment = data.get('수집공정설비', '')
                
                overview = data.get('제조AI데이터셋개요', {})
                background = overview.get('제조데이터셋 구축 배경', '')
                method = overview.get('제조데이터셋 수집 방법', '')
                variables = overview.get('제조데이터셋 주요 변수', '')
                usage = overview.get('제조데이터셋 활용 방법', '')
                
                collection = data.get('제조AI데이터셋수집방법', {})
                
                content = f"""# {title}

## 업체 정보
- 업체명: {company}
- 수집업종: {industry}
- 수집목적: {purpose}
- 수집공정설비: {equipment}

## 데이터셋 구축 배경
{background}

## 데이터셋 수집 방법
{method}

## 주요 변수
{variables}

## 데이터셋 활용 방법
{usage}

## 수집 상세
- 업종: {collection.get('업종', '')}
- 제조공장명: {collection.get('제조공장명', '')}
- 수집장비: {collection.get('수집장비', '')}
- 수집기간: {collection.get('수집기간', '')}
- 수집주기: {collection.get('수집주기', '')}
"""
                doc = Document(
                    page_content=content,
                    metadata={
                        'source': str(jf),
                        'company': company,
                        'title': title,
                        'industry': industry,
                        'type': 'KAMP_게시물'
                    }
                )
                documents.append(doc)
            except Exception as e:
                print(f"   ⚠️ {jf.name} 로드 실패: {e}")
        
        if documents:
            data_source = 'KAMP JSON'
            print(f"✅ KAMP JSON에서 {len(documents)}개 문서 생성 완료")

# ---- Fallback: 기존 샘플 문서 생성 ----
if not documents:
    data_source = 'Fallback (샘플 마크다운)'
    print("⚠️ KAMP 데이터를 찾을 수 없습니다. 기본 샘플 문서를 생성합니다.\n")
    
    # 문서 디렉토리 생성
    docs_dir = Path('../docs/manufacturing')
    docs_dir.mkdir(exist_ok=True, parents=True)
    
    sample_docs = {
        'predictive_maintenance.md': """# 예지보전 시스템

## 개요
예지보전(Predictive Maintenance)은 설비의 상태를 실시간으로 모니터링하여 고장을 사전에 예측하는 기술입니다.

## 주요 기술
- **진동 분석**: 설비의 진동 패턴을 분석하여 이상 징후 탐지
- **온도 모니터링**: 과열 여부를 실시간 감시
- **음향 분석**: 비정상 소음 패턴 식별

## 적용 분야
- 회전 기계 (모터, 펌프, 팬)
- 베어링 및 기어박스
- 컴프레서 및 냉각 시스템

## 효과
- 예상치 못한 설비 다운타임 80% 감소
- 유지보수 비용 30% 절감
- 설비 수명 25% 연장""",
        'quality_control.md': """# 품질 관리 시스템

## 개요
제조 품질 관리는 불량품을 조기에 발견하고 원인을 분석하여 품질을 개선하는 프로세스입니다.

## 검사 방법
- **육안 검사**: 외관 불량 육안 확인
- **머신비전**: AI 기반 자동 불량 검출
- **치수 측정**: 정밀 측정 장비를 활용한 규격 검사

## 불량 유형
1. **스크래치**: 표면 긁힘 흠집
2. **오염**: 이물질 부착 및 얼룩
3. **균열**: 표면 또는 내부 크랙
4. **치수 불량**: 규격 초과 또는 미달

## 품질 지표
- 불량률: 전체 생산량 대비 불량품 비율 (<1% 목표)
- First Pass Yield (FPY): 첫 검사 합격률 (>98% 목표)
- 공정 능력 지수 (Cpk): 공정 안정성 지표 (>1.33 목표)""",
        'defect_classification.md': """# AI 기반 불량 분류

## 개요
딥러닝을 활용한 자동 불량 분류 시스템으로 검사 속도와 정확도를 향상시킵니다.

## 주요 알고리즘
- **CNN (Convolutional Neural Network)**: 이미지 특징 추출
- **Vision Transformer (ViT)**: 글로벌 패턴 학습
- **YOLO**: 실시간 객체 탐지 및 위치 파악

## 학습 데이터
- 정상 이미지: 10,000장
- 불량 이미지: 3,000장 (스크래치, 오염, 균열 등)
- 데이터 증강: 회전, 반전, 색상 조정으로 3배 확장

## 성능 목표
- 정확도 (Accuracy): 95% 이상
- 재현율 (Recall): 90% 이상 (불량 미검출 최소화)
- 추론 속도: 이미지당 100ms 이내

## 배포 환경
- Edge Device: NVIDIA Jetson Xavier NX
- 프레임워크: PyTorch + ONNX Runtime
- 카메라: 산업용 GigE Vision 카메라 (5MP, 30fps)""",
        'smart_factory.md': """# 스마트 팩토리 구축

## 개요
IoT, AI, 빅데이터 기술을 융합하여 지능형 제조 시스템을 구축합니다.

## 핵심 기술
- **IoT 센서**: 온도, 압력, 진동, 전력 등 실시간 데이터 수집
- **MES (Manufacturing Execution System)**: 생산 실행 관리
- **Digital Twin**: 실제 공장의 가상 모델 구현
- **AI 예측 모델**: 수요 예측, 재고 최적화

## 구현 단계
1. 설비 연결성 확보 (OPC-UA, MQTT)
2. 데이터 수집 및 저장 (Time-series DB)
3. 실시간 모니터링 대시보드 구축
4. AI 모델 개발 및 배포
5. 피드백 루프를 통한 지속적 개선

## 기대 효과
- 생산성 30% 향상
- 불량률 50% 감소
- 에너지 비용 20% 절감
- 리드타임 40% 단축"""
    }
    
    for filename, content in sample_docs.items():
        file_path = docs_dir / filename
        file_path.write_text(content.strip(), encoding='utf-8')
    
    print(f"   {len(sample_docs)}개 샘플 문서 생성됨: {docs_dir}")

print(f"\n📊 데이터 소스: {data_source}")
print(f"📊 로드된 문서: {len(documents)}개" if documents else "📊 Fallback 문서 사용 예정")

## 📖 Step 3: 문서 로딩 및 청킹

**Chunking 전략**: RecursiveCharacterTextSplitter (500 chars, 200 overlap)

In [ ]:
# 문서 로더 - KAMP 데이터가 이미 로드된 경우 스킵
if not documents:
    # Fallback: 샘플 마크다운 디렉토리에서 로드
    docs_dir = Path('../docs/manufacturing')
    
    loader = DirectoryLoader(
        str(docs_dir),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'}
    )
    documents = loader.load()
    print(f"📚 Fallback 문서 디렉토리에서 로드: {len(documents)}개")
else:
    print(f"📚 KAMP 데이터에서 이미 로드됨: {len(documents)}개")

for idx, doc in enumerate(documents):
    source = doc.metadata.get('source', 'Unknown')
    source_name = Path(source).name if '/' in source or '\\' in source else source
    print(f"   [{idx+1}] {source_name} - {len(doc.page_content)} chars")

In [ ]:
# Text Splitter 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 청크 크기
    chunk_overlap=200,  # 중복 영역
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 문서 분할
chunks = text_splitter.split_documents(documents)

print(f"\n🔪 청크 분할 완료!")
print(f"   청크 수: {len(chunks)}개")
print(f"   평균 길이: {np.mean([len(chunk.page_content) for chunk in chunks]):.1f} chars")
print(f"   최소/최대: {min([len(chunk.page_content) for chunk in chunks])}/{max([len(chunk.page_content) for chunk in chunks])} chars")

# 샘플 청크 출력
print(f"\n📄 샘플 청크 (첫 번째):")
print(f"   길이: {len(chunks[0].page_content)} chars")
print(f"   출처: {Path(chunks[0].metadata['source']).name}")
print(f"   내용: {chunks[0].page_content[:200]}...")

## 🤖 Step 4: 한국어 임베딩 모델 로드

**모델**: jhgan/ko-sroberta-multitask (KoSRoBERTa)

In [ ]:
print("🔄 한국어 임베딩 모델 로드 중...\n")

# HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sroberta-multitask',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ 임베딩 모델 로드 완료!")
print(f"   모델: jhgan/ko-sroberta-multitask")

# 임베딩 차원 확인
sample_embedding = embeddings.embed_query("테스트 문장")
print(f"   임베딩 차원: {len(sample_embedding)}")
print(f"   벡터 범위: [{min(sample_embedding):.4f}, {max(sample_embedding):.4f}]")

## 💾 Step 5: ChromaDB 벡터 DB 구축

**Persistent Storage**: 디스크에 영구 저장

In [ ]:
print("🔄 ChromaDB 벡터 DB 생성 중...\n")

# ChromaDB 저장 경로
chroma_db_dir = Path('../chroma_db')
chroma_db_dir.mkdir(exist_ok=True)

# ChromaDB 생성 (Persistent)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(chroma_db_dir),
    collection_name='manufacturing_docs'
)

# 영구 저장
vectorstore.persist()

print("✅ ChromaDB 생성 완료!")
print(f"   저장 경로: {chroma_db_dir}")
print(f"   컬렉션: manufacturing_docs")
print(f"   문서 수: {vectorstore._collection.count()}개")

## 🔍 Step 6: Semantic Search 테스트

**Top-K Retrieval**: 질의와 가장 유사한 문서 검색

In [ ]:
# 테스트 질의
test_queries = [
    "예지보전이란 무엇인가요?",
    "불량 유형에는 어떤 것들이 있나요?",
    "AI로 불량을 분류하는 방법은?",
    "스마트 팩토리의 기대 효과는?"
]

print("🔍 Semantic Search 테스트\n")

for idx, query in enumerate(test_queries, 1):
    print(f"{'='*80}")
    print(f"[질의 {idx}] {query}")
    print(f"{'='*80}")
    
    # 유사 문서 검색 (Top-3)
    results = vectorstore.similarity_search_with_score(query, k=3)
    
    for rank, (doc, score) in enumerate(results, 1):
        source = Path(doc.metadata['source']).name
        print(f"\n[순위 {rank}] 유사도: {score:.4f} | 출처: {source}")
        print(f"   {doc.page_content[:150]}...")
    
    print()

## 🤝 Step 7: 하이브리드 검색 (BM25 + Semantic)

**전략**: 키워드 검색(BM25) + 의미 검색(Semantic) 결합

In [ ]:
print("🔄 BM25 인덱스 구축 중...\n")

# 청크 텍스트 추출
chunk_texts = [chunk.page_content for chunk in chunks]

# 토크나이저 (간단한 공백 기반)
tokenized_corpus = [text.split() for text in chunk_texts]

# BM25 인덱스 생성
bm25 = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 인덱스 생성 완료!")
print(f"   문서 수: {len(tokenized_corpus)}개")
print(f"   평균 토큰 수: {np.mean([len(tokens) for tokens in tokenized_corpus]):.1f}")

In [ ]:
def hybrid_search(query, k=3, alpha=0.5):
    """
    하이브리드 검색: BM25 + Semantic
    
    Args:
        query: 질의 문자열
        k: 검색할 문서 수
        alpha: BM25 가중치 (0-1, 높을수록 BM25 중시)
    """
    # BM25 검색
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-10)
    
    # Semantic 검색
    semantic_results = vectorstore.similarity_search_with_score(query, k=len(chunks))
    semantic_scores = np.array([1 / (1 + score) for doc, score in semantic_results])  # 거리 → 유사도
    semantic_scores = (semantic_scores - semantic_scores.min()) / (semantic_scores.max() - semantic_scores.min() + 1e-10)
    
    # 하이브리드 점수 계산
    hybrid_scores = alpha * bm25_scores + (1 - alpha) * semantic_scores
    
    # Top-K 선택
    top_k_indices = np.argsort(hybrid_scores)[::-1][:k]
    
    results = []
    for idx in top_k_indices:
        results.append({
            'chunk': chunks[idx],
            'bm25_score': bm25_scores[idx],
            'semantic_score': semantic_scores[idx],
            'hybrid_score': hybrid_scores[idx]
        })
    
    return results

# 하이브리드 검색 테스트
query = "AI 불량 분류의 성능 목표는?"

print(f"🔍 하이브리드 검색 테스트\n")
print(f"질의: {query}\n")

hybrid_results = hybrid_search(query, k=3, alpha=0.5)

for rank, result in enumerate(hybrid_results, 1):
    source = Path(result['chunk'].metadata['source']).name
    print(f"[순위 {rank}]")
    print(f"   출처: {source}")
    print(f"   BM25: {result['bm25_score']:.4f} | Semantic: {result['semantic_score']:.4f} | Hybrid: {result['hybrid_score']:.4f}")
    print(f"   {result['chunk'].page_content[:150]}...\n")

## 🎯 Step 8: RAG 파이프라인 구축

**LangChain QA Chain**: Retriever + LLM (시뮬레이션)

In [ ]:
# Retriever 설정
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ Retriever 생성 완료!")
print(f"   검색 방식: Similarity")
print(f"   Top-K: 3")

# 프롬프트 템플릿
prompt_template = """
당신은 제조 분야 전문가입니다. 아래 문서를 참고하여 질문에 답변하세요.

문서:
{context}

질문: {question}

답변:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print(f"\n✅ 프롬프트 템플릿 생성 완료!")

In [ ]:
# RAG 시뮬레이션 함수
def simulate_rag(query):
    """
    RAG 시뮬레이션 (LLM 없이 검색된 문서 반환)
    
    실제 배포 시에는 LLM (GPT, Claude, Llama 등)을 연결하여 답변 생성
    """
    # 관련 문서 검색
    retrieved_docs = retriever.get_relevant_documents(query)
    
    # Context 생성
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # 프롬프트 생성
    full_prompt = PROMPT.format(context=context, question=query)
    
    return {
        'query': query,
        'retrieved_docs': retrieved_docs,
        'context': context,
        'prompt': full_prompt
    }

# RAG 테스트
test_query = "예지보전 시스템의 효과는 무엇인가요?"

print(f"🤖 RAG 시뮬레이션\n")
print(f"질의: {test_query}\n")

rag_result = simulate_rag(test_query)

print(f"📚 검색된 문서: {len(rag_result['retrieved_docs'])}개\n")
for idx, doc in enumerate(rag_result['retrieved_docs'], 1):
    source = Path(doc.metadata['source']).name
    print(f"[문서 {idx}] {source}")
    print(f"   {doc.page_content[:100]}...\n")

print(f"💬 생성된 프롬프트:")
print(f"{rag_result['prompt'][:500]}...")

print(f"\n💡 실제 배포 시:")
print(f"   - 위 프롬프트를 LLM (GPT-4, Claude, Llama 등)에 입력")
print(f"   - LLM이 문서 기반 답변 생성")
print(f"   - 할루시네이션 감소, 정확도 향상")

## 📊 Step 9: RAG 평가 메트릭

**평가 지표**: Context Relevance, Answer Similarity

In [ ]:
# 평가용 QA 쌍
eval_qa_pairs = [
    {
        'question': '예지보전의 주요 기술은?',
        'expected_keywords': ['진동 분석', '온도 모니터링', '음향 분석']
    },
    {
        'question': '불량 유형 4가지는?',
        'expected_keywords': ['스크래치', '오염', '균열', '치수']
    },
    {
        'question': 'AI 불량 분류의 정확도 목표는?',
        'expected_keywords': ['95%', '정확도']
    },
    {
        'question': '스마트 팩토리의 생산성 향상률은?',
        'expected_keywords': ['30%', '생산성']
    }
]

print("📊 RAG 평가\n")

# Context Relevance 평가
evaluation_results = []

for qa in eval_qa_pairs:
    question = qa['question']
    expected_keywords = qa['expected_keywords']
    
    # 문서 검색
    retrieved_docs = retriever.get_relevant_documents(question)
    context = " ".join([doc.page_content for doc in retrieved_docs])
    
    # Keyword 매칭
    found_keywords = [kw for kw in expected_keywords if kw in context]
    relevance_score = len(found_keywords) / len(expected_keywords)
    
    evaluation_results.append({
        'question': question,
        'expected_keywords': expected_keywords,
        'found_keywords': found_keywords,
        'relevance_score': relevance_score
    })
    
    print(f"질문: {question}")
    print(f"   기대 키워드: {expected_keywords}")
    print(f"   발견 키워드: {found_keywords}")
    print(f"   Context Relevance: {relevance_score:.2%}\n")

# 전체 평균
avg_relevance = np.mean([result['relevance_score'] for result in evaluation_results])
print(f"📈 평균 Context Relevance: {avg_relevance:.2%}")

## 💾 Step 10: 결과 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 1. 청크 정보
chunk_info = pd.DataFrame({
    'chunk_id': range(len(chunks)),
    'source': [Path(chunk.metadata['source']).name for chunk in chunks],
    'length': [len(chunk.page_content) for chunk in chunks],
    'preview': [chunk.page_content[:100] for chunk in chunks]
})
chunk_file = output_dir / '02_chunks_info.csv'
chunk_info.to_csv(chunk_file, index=False, encoding='utf-8-sig')
print(f"✅ 청크 정보 저장: {chunk_file}")

# 2. 평가 결과
eval_df = pd.DataFrame(evaluation_results)
eval_file = output_dir / '02_rag_evaluation.csv'
eval_df.to_csv(eval_file, index=False, encoding='utf-8-sig')
print(f"✅ 평가 결과 저장: {eval_file}")

# 3. 검색 테스트 결과
search_results = []
for query in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    for rank, (doc, score) in enumerate(results, 1):
        search_results.append({
            'query': query,
            'rank': rank,
            'source': Path(doc.metadata['source']).name,
            'similarity_score': score,
            'content_preview': doc.page_content[:100]
        })

search_df = pd.DataFrame(search_results)
search_file = output_dir / '02_search_results.csv'
search_df.to_csv(search_file, index=False, encoding='utf-8-sig')
print(f"✅ 검색 결과 저장: {search_file}")

# 4. RAG 설정 정보
rag_config = {
    'embedding_model': 'jhgan/ko-sroberta-multitask',
    'embedding_dim': len(sample_embedding),
    'vectorstore': 'ChromaDB',
    'collection': 'manufacturing_docs',
    'chunk_size': 500,
    'chunk_overlap': 200,
    'total_chunks': len(chunks),
    'retrieval_k': 3,
    'avg_relevance_score': avg_relevance
}

config_file = output_dir / '02_rag_config.json'
with open(config_file, 'w', encoding='utf-8') as f:
    json.dump(rag_config, f, indent=2, ensure_ascii=False)
print(f"✅ RAG 설정 저장: {config_file}")

print("\n🎉 모든 결과 저장 완료!")

---

## 🎯 학습 정리

### ✅ 완료한 내용

1. **문서 처리** - 제조 문서 로딩 및 청킹 (RecursiveCharacterTextSplitter)
2. **한국어 임베딩** - KoSRoBERTa 모델 활용
3. **벡터 DB** - ChromaDB Persistent Storage 구축
4. **Semantic Search** - Top-K 유사도 기반 검색
5. **하이브리드 검색** - BM25 + Semantic 결합
6. **RAG 파이프라인** - LangChain Retriever + 프롬프트 생성
7. **평가** - Context Relevance 측정

---

### 💡 핵심 인사이트

#### RAG의 장점

**기존 LLM vs RAG**:
- **할루시네이션 감소**: 실제 문서 기반 답변으로 정확도 향상
- **최신 정보 반영**: 모델 재학습 없이 문서 업데이트만으로 최신 정보 제공
- **도메인 특화**: 제조 분야 전문 지식을 효과적으로 활용
- **출처 추적**: 답변의 근거가 되는 문서 명시 가능

---

#### 청킹 전략

**Chunk Size vs Overlap**:
- **작은 청크 (200-500)**: 검색 정확도 높음, 문맥 손실 가능성
- **큰 청크 (1000-2000)**: 문맥 보존 좋음, 노이즈 증가
- **Overlap 200**: 문장 경계 보존, 검색 성능 향상

**Separator 전략**:
```python
separators=["\n\n", "\n", ". ", " ", ""]
```
- 단락 → 문장 → 단어 순서로 자연스러운 분할

---

#### 하이브리드 검색

**BM25 vs Semantic**:
- **BM25**: 키워드 정확 매칭, 용어 중요도 계산
- **Semantic**: 의미적 유사도, 동의어 처리
- **하이브리드**: 두 방법의 장점 결합 (alpha 조정)

**Alpha 튜닝**:
- alpha=0.7: 키워드 중시 (기술 용어 많은 경우)
- alpha=0.3: 의미 중시 (자연어 질문 많은 경우)
- alpha=0.5: 균형 (기본값)

---

#### 실무 배포 가이드

**LLM 선택**:
- **Cloud API**: OpenAI GPT-4, Anthropic Claude (고성능, 비용 높음)
- **Self-hosted**: Llama 2, Mistral (비용 낮음, 인프라 필요)
- **Edge Deployment**: Quantized Llama 7B (제한된 하드웨어)

**성능 최적화**:
1. **캐싱**: 자주 묻는 질문 캐싱으로 응답 속도 향상
2. **Re-ranking**: 검색 후 재순위화로 정확도 개선
3. **Compression**: 문서 압축으로 토큰 사용량 감소
4. **Parallel Retrieval**: 다중 인덱스 병렬 검색

**보안 고려사항**:
- 민감 정보 마스킹
- 접근 권한 관리 (문서별 권한)
- 검색 로그 모니터링

---

### 📚 다음 단계

- **Part 3-1**: 제조 챗봇 구축 (RAG + Conversational Memory)
- **Part 3-2**: Multi-modal RAG (이미지 + 텍스트)

### 🔗 참고 자료

- [LangChain 공식 문서](https://python.langchain.com/docs/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [RAG 논문](https://arxiv.org/abs/2005.11401) - Retrieval-Augmented Generation

---

*제조AI 교육 v12 Enhanced | Part 3-1 | 2025.02*